# **Retrain ResNet-50 on Mushroom B**

In [1]:
%%capture
!pip install torchmetrics
!pip install pkbar
!pip install calflops
!pip install torchinfo
!pip install grad-cam
%pip install -q torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Section 1: Config**

In [3]:
import os
import sys
import zipfile
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import v2, InterpolationMode
import timm
from functools import partial

drive_dir = '/content/drive/MyDrive/projectMushroom'
dataset_name = 'MushroomB'
model_name_path = 'resnet50_224_r1_v1'
model_name = 'ResNet50'
TARGET_FOLD = 4
TOL = 1e-6
METRICS = ['loss', 'accuracy', 'f1', 'precision', 'recall']

dataset_drive_dir = f'{drive_dir}/data/{dataset_name}'
orig_dir = f'{drive_dir}/modelWeights/{dataset_name}/{model_name_path}_{dataset_name}'
retrain_dir = f'{drive_dir}/addition/retrain/{model_name_path}_{dataset_name}'
if os.path.exists(retrain_dir) and os.listdir(retrain_dir):
    raise RuntimeError(f'[Error] {retrain_dir} is not empty. Move or delete it before retraining.')
os.makedirs(retrain_dir, exist_ok=True)

os.chdir(drive_dir)
import src.reproducibility as reproducibility
import src.utils as utils
import src.datasets as src_datasets
import src.train as train
import models.models as models

device = torch.device('cuda')
assert torch.cuda.is_available(), '[Error] GPU runtime is required.'
gpu_name = torch.cuda.get_device_name(0)
assert 'L4' in gpu_name, f'[Error] The original model was trained on NVIDIA L4, not {gpu_name}.'
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
       'timm': timm.__version__, 'gpu': gpu_name}
print(ENV)

def callbacks_fn(n_fold, fold_dir):
    checkpoint_path = f'{fold_dir}/{model_name_path}_{dataset_name}_best_weight.pth'
    return [utils.ModelCheckpoint(save_path=checkpoint_path, monitor=train.train_config['monitor'], mode=train.train_config['monitor_mode'])]

/usr/local/lib/python3.13/dist-packages/pkbar/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


{'python': '3.13.15', 'torch': '2.10.0+cu128', 'torchvision': '0.25.0+cu128', 'timm': '1.0.30', 'gpu': 'NVIDIA L4'}


## **Section 2: Load Dataset**

In [4]:
runtime_dir = '/content'
dataset_dir = f'{runtime_dir}/{dataset_name}'
os.makedirs(dataset_dir, exist_ok=True)
with zipfile.ZipFile(f'{dataset_drive_dir}/{dataset_name}_SPLIT.zip', 'r') as zip_ref:
    zip_ref.extractall(dataset_dir)

NUM_CLASSES = len(set(os.listdir(f'{dataset_dir}/remain')) | set(os.listdir(f'{dataset_dir}/test')))
print(f'Number of classes: {NUM_CLASSES}')

Number of classes: 16


## **Section 3: Model**

In [5]:
model_fn = partial(
    models.build_baseline_model,
    model_name=model_name,
    num_classes=NUM_CLASSES,
    device=device,
    freeze_backbone=False
)
model = model_fn()

criterion = nn.CrossEntropyLoss()
optimizer_fn = lambda m : optim.AdamW(m.parameters(), lr=train.train_config['base_lr'], weight_decay=train.train_config['weight_decay'])
scheduler_fn = partial(optim.lr_scheduler.CosineAnnealingLR, T_max=train.train_config['max_epochs'])

model_config = models.get_transform_config(model)

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

## **Section 4: Dataset and Seed**

In [6]:
img_size = model_config['img_size']
resize_size = int(img_size[0] / model_config['crop_pct'])
interp_mode = getattr(InterpolationMode, model_config['interpolation'].upper())
num_workers = min(4, os.cpu_count())
assert num_workers == 4, f'[Error] The original run used 4 workers, this runtime gives {num_workers}.'

reproducibility.global_seed(seed=train.train_config['seed'])
reproducibility.training_seed(seed=train.train_config['seed'])
g = reproducibility.get_generator(seed=train.train_config['seed'])

train_transform = v2.Compose([
    v2.RandomResizedCrop(img_size, scale=(0.3, 1.0), interpolation=interp_mode),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandAugment(num_ops=2, magnitude=6, interpolation=interp_mode),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=model_config['mean'], std=model_config['std']),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False)
])

transform = v2.Compose([
    v2.Resize(resize_size, interpolation=interp_mode, max_size=None, antialias=True),
    v2.CenterCrop(img_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=model_config['mean'], std=model_config['std'])
])

dataloader_kwargs = {
    'batch_size': train.train_config['batch_size'],
    'num_workers': num_workers,
    'worker_init_fn': reproducibility.seed_worker,
    'generator': g,
    'pin_memory': True
}

datasets = src_datasets.Datasets(
    dataset_dir=dataset_dir,
    train_transform=train_transform,
    transform=transform,
    dataloader_kwargs=dataloader_kwargs,
)
datasets.print_summary()
test_loader = datasets.get_test_loader()

Global Seed set to: 42
Training Seed set to: 42
--------------------------------------------------
Dataset Summary
--------------------------------------------------
Class names       : ['Amanita_Caesarea-Edible', 'Amanita_Citrina-Edible', 'Amanita_Pantherina-NotEdible', 'Boletus_Regius-Edible', 'Clitocybe_Costata-Edible', 'Entoloma_Lividum-NotEdible', 'Gyromitra_Esculenta-NotEdible', 'Helvella_Crispa-Edible', 'Hydnum_Rufescens-NotEdible', 'Hygrophorus_Latitabundus-Edible', 'Morchella_Deliciosa-Edible', 'Omphalotus_Olearius-NotEdible', 'Phallus_Impudicus-NotEdible', 'Rubroboletus_Satanas-NotEdible', 'Russula_Cyanoxantha-Edible', 'Russula_Delica-NotEdible']
Class to Index    : {'Amanita_Caesarea-Edible': 0, 'Amanita_Citrina-Edible': 1, 'Amanita_Pantherina-NotEdible': 2, 'Boletus_Regius-Edible': 3, 'Clitocybe_Costata-Edible': 4, 'Entoloma_Lividum-NotEdible': 5, 'Gyromitra_Esculenta-NotEdible': 6, 'Helvella_Crispa-Edible': 7, 'Hydnum_Rufescens-NotEdible': 8, 'Hygrophorus_Latitabundus-Edib

## **Section 5: Train**

In [7]:
trainer = train.KFoldTrainer(
    n_splits=train.train_config['n_folds'],
    shuffle=True,
    random_state=train.train_config['seed'],
    model_name=model_name_path,
    dataset_name=dataset_name,
    base_dir=retrain_dir,
    first_metric=train.train_config['first_metric'],
    first_metric_mode=train.train_config['first_metric_mode'],
    secondary_metric=train.train_config['secondary_metric'],
    secondary_metric_mode=train.train_config['secondary_metric_mode'],
    device=device,
    criterion=criterion,
    n_classes=NUM_CLASSES,
    task=train.train_config['task'],
    average=train.train_config['average'],
    max_grad_norm=0.0
)

fold_histories = trainer.fit_kfold(
    train_dataset=datasets.train_dataset,
    val_dataset=datasets.val_dataset,
    labels=datasets.all_classes,
    model_fn=model_fn,
    optimizer_fn=optimizer_fn,
    epochs=train.train_config['max_epochs'],
    dataloader_kwargs=dataloader_kwargs,
    callbacks_fn=callbacks_fn,
    scheduler_fn=scheduler_fn
)

[Logs] Setup directories at '/content/drive/MyDrive/projectMushroom/addition/retrain/resnet50_224_r1_v1_MushroomB' successfully.
Starting Stratified 5-Fold Cross Validation

--------------------------------------------- FOLD 1/5 ---------------------------------------------
Epoch: 1/30
60/60 [==============================] - 14s 237ms/step - loss: 2.6978 - lr: 1.0000e-04 - accuracy: 0.1574 - f1: 0.0574 - precision: 0.1623 - recall: 0.0967
Epoch 1/30 | val_loss: 2.5555 - val_accuracy: 0.2750 - val_f1: 0.1391 - val_precision: 0.2232 - val_recall: 0.1778
[Logs] Epoch: 1 val_accuracy improved from -inf to 0.2750, saving model to /content/drive/MyDrive/projectMushroom/addition/retrain/resnet50_224_r1_v1_MushroomB/fold_1/resnet50_224_r1_v1_MushroomB_best_weight.pth
Epoch: 2/30
60/60 [==============================] - 13s 222ms/step - loss: 2.3975 - lr: 9.9726e-05 - accuracy: 0.3106 - f1: 0.1682 - precision: 0.4475 - recall: 0.2062
Epoch 2/30 | val_loss: 1.9799 - val_accuracy: 0.4250 - val_f

## **Section 6: Evaluate**

In [8]:
trainer.evaluate_kfold(
    val_dataset=datasets.val_dataset,
    labels=datasets.all_classes,
    dataloader_kwargs=dataloader_kwargs,
    model_fn=model_fn,
    weight_type=train.train_config['weight_type']
)
trainer.predict_kfold(
    dataloader=test_loader,
    model_fn=model_fn,
    weight_type=train.train_config['weight_type'],
    timer_report_dict=False,
    verbose=1
)

Evaluating 5-Fold Models on their respective Validation Sets (BEST weights)

[*] Evaluating Fold 1/5 on its Validation Set...
15/15 [==============================] - 3s 210ms/step - loss: 0.1051 - accuracy: 0.9792 - f1: 0.9757 - precision: 0.9760 - recall: 0.9763
[*] Evaluating Fold 2/5 on its Validation Set...
15/15 [==============================] - 4s 258ms/step - loss: 0.1160 - accuracy: 0.9708 - f1: 0.9639 - precision: 0.9645 - recall: 0.9641
[*] Evaluating Fold 3/5 on its Validation Set...
15/15 [==============================] - 4s 286ms/step - loss: 0.0725 - accuracy: 0.9854 - f1: 0.9850 - precision: 0.9861 - recall: 0.9845
[*] Evaluating Fold 4/5 on its Validation Set...
15/15 [==============================] - 3s 232ms/step - loss: 0.0766 - accuracy: 0.9917 - f1: 0.9905 - precision: 0.9914 - recall: 0.9896
[*] Evaluating Fold 5/5 on its Validation Set...
15/15 [==============================] - 4s 268ms/step - loss: 0.0765 - accuracy: 0.9833 - f1: 0.9793 - precision: 0.9784 

array([[9.5752108e-01, 2.6832197e-03, 4.7304467e-03, ..., 8.0460776e-03,
        1.6253944e-03, 6.7936227e-04],
       [9.8031729e-01, 2.1589380e-03, 2.7859674e-03, ..., 1.5699361e-03,
        7.6273491e-04, 4.2274286e-04],
       [9.8611605e-01, 1.7528509e-03, 9.4049162e-04, ..., 4.6514711e-04,
        2.1063322e-04, 9.4301671e-05],
       ...,
       [1.1997012e-04, 2.1634126e-04, 2.3988029e-04, ..., 3.1352130e-05,
        6.3709379e-03, 9.9018037e-01],
       [3.4638878e-04, 6.9120614e-04, 8.9349772e-04, ..., 4.2231366e-04,
        6.9712824e-03, 9.8611534e-01],
       [9.0848494e-05, 6.1720499e-04, 1.0304141e-03, ..., 2.5413843e-05,
        3.9447141e-03, 9.8837870e-01]], dtype=float32)

## **Section 7: Compare with the Original Run**

In [9]:
def fold_files(base, k):
    return {
        'history': f'{base}/fold_{k}/fold_{k}_training_history.csv',
        'validate': f'{base}/fold_{k}/fold_{k}_validate_perf_best_weight.csv',
        'test_probs': f'{base}/fold_{k}/fold_{k}_test_probs_best_weight.csv',
        'weight': f'{base}/fold_{k}/{model_name_path}_{dataset_name}_best_weight.pth',
    }

def read(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'[Error] File not found: {path}')
    return pd.read_csv(path, float_precision='round_trip')

rows = []
for k in range(1, train.train_config['n_folds'] + 1):
    o, n = fold_files(orig_dir, k), fold_files(retrain_dir, k)
    oh, nh = read(o['history']), read(n['history'])
    if list(oh.columns) != list(nh.columns) or oh.shape != nh.shape:
        raise AssertionError(f'[Error] Fold {k}: training history format differs ({oh.shape} vs {nh.shape})')
    ov, nv = read(o['validate']).iloc[0], read(n['validate']).iloc[0]
    op, npb = read(o['test_probs']).values, read(n['test_probs']).values
    if op.shape != npb.shape:
        raise AssertionError(f'[Error] Fold {k}: test probability shape differs ({op.shape} vs {npb.shape})')
    rows.append({
        'fold': k,
        'epochs': len(nh),
        'history_max_diff': float(np.abs(oh.values.astype(float) - nh.values.astype(float)).max()),
        'validate_max_diff': float(max(abs(ov[m] - nv[m]) for m in METRICS)),
        'test_prob_max_diff': float(np.abs(op - npb).max()),
        'same_prediction': int((op.argmax(axis=1) == npb.argmax(axis=1)).sum()),
        'n_test': len(op),
        'weight_bytes': os.path.getsize(n['weight']),
        'weight_is_zip': zipfile.is_zipfile(n['weight']),
    })

compare_df = pd.DataFrame(rows)
compare_df['passed'] = ((compare_df['history_max_diff'] <= TOL) & (compare_df['validate_max_diff'] <= TOL)
                        & (compare_df['test_prob_max_diff'] <= TOL) & (compare_df['same_prediction'] == compare_df['n_test'])
                        & (compare_df['weight_bytes'] > 0) & compare_df['weight_is_zip'])
for key, value in ENV.items():
    compare_df[key] = value
compare_df.to_csv(f'{retrain_dir}/verification.csv', index=False)
display(compare_df)

if compare_df.set_index('fold').loc[TARGET_FOLD, 'passed']:
    print(f'[Logs] PASSED: fold {TARGET_FOLD} reproduces the original run within {TOL}')
else:
    print(f'[Logs] FAILED: fold {TARGET_FOLD} does not reproduce the original run. Do not use this checkpoint.')

,fold,epochs,history_max_diff,validate_max_diff,test_prob_max_diff,same_prediction,n_test,weight_bytes,weight_is_zip,passed,python,torch,torchvision,timm,gpu
0,1,30,0.0,0.0,0.0,600,600,282973634,True,True,3.13.15,2.10.0+cu128,0.25.0+cu128,1.0.30,NVIDIA L4
1,2,30,0.0,0.0,0.0,600,600,282973634,True,True,3.13.15,2.10.0+cu128,0.25.0+cu128,1.0.30,NVIDIA L4
2,3,30,0.0,0.0,0.0,600,600,282973634,True,True,3.13.15,2.10.0+cu128,0.25.0+cu128,1.0.30,NVIDIA L4
3,4,30,0.0,0.0,0.0,600,600,282973634,True,True,3.13.15,2.10.0+cu128,0.25.0+cu128,1.0.30,NVIDIA L4
4,5,30,0.0,0.0,0.0,600,600,282973634,True,True,3.13.15,2.10.0+cu128,0.25.0+cu128,1.0.30,NVIDIA L4


[Logs] PASSED: fold 4 reproduces the original run within 1e-06


In [10]:
drive.flush_and_unmount()